# Networking RAG - Phase 4 Improvements

This notebook improves the Networking RAG system by implementing two retrieval enhancements:

1. Better Chunking
2. Hybrid Search

The objective is to improve retrieval quality and answer relevance while maintaining the existing LangGraph-based architecture.

In [3]:
# Install required libraries

try:

    !pip install -q \
    chromadb \
    wikipedia-api \
    langchain-text-splitters \
    google-genai \
    rank-bm25 \
    langgraph \
    groq

    print(
        "Libraries installed successfully!"
    )

except Exception as e:

    print(
        f"Installation Error: {e}"
    )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.

In [4]:
# Import required libraries

import os
import pickle
import chromadb
import wikipediaapi

from typing import TypedDict

from google import genai
from groq import Groq

from rank_bm25 import BM25Okapi

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from google.colab import userdata

In [5]:
# Initialize API clients

try:

    gemini_client = genai.Client(
        api_key=userdata.get(
            "GEMINI_API_KEY"
        )
    )

    groq_client = Groq(
        api_key=userdata.get(
            "GROQ_API_KEY"
        )
    )

    print(
        "API clients initialized successfully!"
    )

except Exception as e:

    print(
        f"Initialization Error: {e}"
    )

API clients initialized successfully!


In [6]:
# Download networking documents

try:

    wiki = wikipediaapi.Wikipedia(
        language="en",
        user_agent="networking-rag-phase4"
    )

    topics = [
        "Transmission Control Protocol",
        "User Datagram Protocol",
        "Domain Name System",
        "Internet Protocol",
        "OSI model",
        "TCP/IP model",
        "Routing",
        "Network switch",
        "Computer network",
        "Hypertext Transfer Protocol"
    ]

    os.makedirs(
        "networking_docs",
        exist_ok=True
    )

    for topic in topics:

        page = wiki.page(topic)

        if page.exists():

            filename = (
                topic.replace("/", "_")
                     .replace(" ", "_")
            ) + ".txt"

            with open(
                f"networking_docs/{filename}",
                "w",
                encoding="utf-8"
            ) as f:

                f.write(page.text)

    print(
        "Documents downloaded successfully!"
    )

except Exception as e:

    print(
        f"Download Error: {e}"
    )

Documents downloaded successfully!


In [7]:
# Load networking documents

try:

    documents = []

    folder_path = "networking_docs"

    for file in os.listdir(folder_path):

        if file.endswith(".txt"):

            file_path = os.path.join(
                folder_path,
                file
            )

            with open(
                file_path,
                "r",
                encoding="utf-8"
            ) as f:

                documents.append(
                    f.read()
                )

    print(
        "Documents loaded successfully!"
    )

    print(
        "Number of documents:",
        len(documents)
    )

except Exception as e:

    print(
        f"Loading Error: {e}"
    )

Documents loaded successfully!
Number of documents: 10


In [8]:
# Create improved chunks

try:

    splitter = (
        RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=[
                "\n\n",
                "\n",
                ". ",
                " ",
                ""
            ]
        )
    )

    improved_chunks = []

    for document in documents:

        document_chunks = (
            splitter.split_text(
                document
            )
        )

        improved_chunks.extend(
            document_chunks
        )

    print(
        "Improved chunking completed!"
    )

    print(
        "Total Chunks:",
        len(improved_chunks)
    )

except Exception as e:

    print(
        f"Chunking Error: {e}"
    )

Improved chunking completed!
Total Chunks: 506


In [9]:
# Generate embeddings for improved chunks

try:

    import time

    improved_embeddings = []

    for i, chunk in enumerate(
        improved_chunks
    ):

        if i > 0 and i % 90 == 0:

            print(
                "Rate limit protection activated. Waiting 60 seconds..."
            )

            time.sleep(60)

        response = (
            gemini_client.models.embed_content(
                model="models/gemini-embedding-001",
                contents=chunk
            )
        )

        improved_embeddings.append(
            response.embeddings[0].values
        )

        if (i + 1) % 10 == 0:

            print(
                f"Processed {i + 1} chunks"
            )

    print(
        "Embedding generation completed!"
    )

    print(
        "Total embeddings:",
        len(improved_embeddings)
    )

except Exception as e:

    print(
        f"Embedding Error: {e}"
    )

Processed 10 chunks
Processed 20 chunks
Processed 30 chunks
Processed 40 chunks
Processed 50 chunks
Processed 60 chunks
Processed 70 chunks
Processed 80 chunks
Processed 90 chunks
Rate limit protection activated. Waiting 60 seconds...
Processed 100 chunks
Processed 110 chunks
Processed 120 chunks
Processed 130 chunks
Processed 140 chunks
Processed 150 chunks
Processed 160 chunks
Processed 170 chunks
Processed 180 chunks
Rate limit protection activated. Waiting 60 seconds...
Processed 190 chunks
Processed 200 chunks
Processed 210 chunks
Processed 220 chunks
Processed 230 chunks
Processed 240 chunks
Processed 250 chunks
Processed 260 chunks
Processed 270 chunks
Rate limit protection activated. Waiting 60 seconds...
Processed 280 chunks
Processed 290 chunks
Processed 300 chunks
Processed 310 chunks
Processed 320 chunks
Processed 330 chunks
Processed 340 chunks
Processed 350 chunks
Processed 360 chunks
Rate limit protection activated. Waiting 60 seconds...
Processed 370 chunks
Processed 38

In [10]:
# Store improved chunks in ChromaDB

try:

    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb_phase4"
    )

    collection = (
        chroma_client.get_or_create_collection(
            name="networking_docs_phase4"
        )
    )

    if collection.count() > 0:

        collection.delete(
            ids=collection.get()["ids"]
        )

    collection.add(
        ids=[
            f"chunk_{i}"
            for i in range(
                len(improved_chunks)
            )
        ],
        documents=improved_chunks,
        embeddings=improved_embeddings
    )

    print(
        "Chunks and embeddings stored successfully!"
    )

    print(
        "Total records:",
        collection.count()
    )

except Exception as e:

    print(
        f"ChromaDB Error: {e}"
    )

ChromaDB Error: Unequal lengths for fields: ids: 506, embeddings: 389, documents: 506 in add.


In [24]:
# Create BM25 index

try:

    tokenized_chunks = [

        chunk.lower().split()

        for chunk in improved_chunks

    ]

    bm25 = BM25Okapi(
        tokenized_chunks
    )

    print(
        "BM25 index created successfully!"
    )

    print(
        "Indexed Chunks:",
        len(tokenized_chunks)
    )

except Exception as e:

    print(
        f"BM25 Error: {e}"
    )

BM25 index created successfully!
Indexed Chunks: 506


In [25]:
# Create hybrid retrieval function

try:

    def hybrid_retrieve(
        question,
        top_k=5
    ):

        response = (
            gemini_client.models.embed_content(
                model="models/gemini-embedding-001",
                contents=question
            )
        )

        query_embedding = (
            response.embeddings[0].values
        )

        vector_results = (
            collection.query(
                query_embeddings=[
                    query_embedding
                ],
                n_results=top_k
            )
        )

        vector_chunks = (
            vector_results["documents"][0]
        )

        tokenized_query = (
            question.lower().split()
        )

        bm25_scores = (
            bm25.get_scores(
                tokenized_query
            )
        )

        bm25_indices = (
            sorted(
                range(
                    len(bm25_scores)
                ),
                key=lambda i:
                bm25_scores[i],
                reverse=True
            )[:top_k]
        )

        bm25_chunks = [

            improved_chunks[i]

            for i in bm25_indices

        ]

        combined_chunks = []

        for chunk in (
            vector_chunks +
            bm25_chunks
        ):

            if chunk not in combined_chunks:

                combined_chunks.append(
                    chunk
                )

        return combined_chunks[:top_k]

    print(
        "Hybrid retrieval function created successfully!"
    )

except Exception as e:

    print(
        f"Hybrid Retrieval Error: {e}"
    )

Hybrid retrieval function created successfully!


In [26]:
# Test hybrid retrieval

try:

    results = hybrid_retrieve(
        "What is the purpose of DNS?"
    )

    print(
        "Retrieved Chunks:",
        len(results)
    )

    for i, chunk in enumerate(
        results,
        start=1
    ):

        print("=" * 60)
        print(
            f"Chunk {i}"
        )
        print(
            chunk[:300]
        )
        print()

except Exception as e:

    print(
        f"Testing Error: {e}"
    )

Testing Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 36.901090635s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-embedd

In [14]:
# Create graph state

class GraphState(TypedDict):

    question: str

    retrieved_chunks: list

    answer: str

In [15]:
# Create hybrid retrieval node

try:

    def retrieve_chunks(state):

        state["retrieved_chunks"] = (
            hybrid_retrieve(
                state["question"]
            )
        )

        print(
            "Hybrid retrieval completed!"
        )

        return state

    print(
        "Hybrid Retrieval Node created!"
    )

except Exception as e:

    print(
        f"Node Creation Error: {e}"
    )

Hybrid Retrieval Node created!


In [16]:
# Create answer generation node

try:

    def generate_answer(state):

        context = "\n\n".join(
            state["retrieved_chunks"]
        )

        prompt = f"""
You are a networking assistant.

Answer only using the provided context.

If the answer is not available in the context,
respond with:

I could not find that information in the knowledge base.

Context:
{context}

Question:
{state["question"]}
"""

        response = (
            groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )
        )

        state["answer"] = (
            response.choices[0].message.content
        )

        return state

    print(
        "Generate Answer Node created!"
    )

except Exception as e:

    print(
        f"Node Creation Error: {e}"
    )

Generate Answer Node created!


In [17]:
# Compile LangGraph workflow

try:

    graph_builder = StateGraph(
        GraphState
    )

    graph_builder.add_node(
        "retrieve_chunks",
        retrieve_chunks
    )

    graph_builder.add_node(
        "generate_answer",
        generate_answer
    )

    graph_builder.add_edge(
        START,
        "retrieve_chunks"
    )

    graph_builder.add_edge(
        "retrieve_chunks",
        "generate_answer"
    )

    graph_builder.add_edge(
        "generate_answer",
        END
    )

    graph = (
        graph_builder.compile()
    )

    print(
        "LangGraph compiled successfully!"
    )

except Exception as e:

    print(
        f"Compilation Error: {e}"
    )

LangGraph compiled successfully!


In [18]:
# Test improved RAG pipeline

try:

    result = graph.invoke(
        {
            "question":
            "What is the purpose of DNS?"
        }
    )

    print(
        "\nFinal Answer:\n"
    )

    print(
        result["answer"]
    )

except Exception as e:

    print(
        f"Execution Error: {e}"
    )

Execution Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 48.442511026s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-embe

In [19]:
result = graph.invoke(
    {
        "question":
        "What is the difference between TCP and UDP?"
    }
)

print(result["answer"])

Hybrid retrieval completed!
I could not find that information in the knowledge base.


In [20]:
# Save improved chunks and embeddings

try:

    import pickle

    with open(
        "improved_chunks.pkl",
        "wb"
    ) as f:

        pickle.dump(
            improved_chunks,
            f
        )

    with open(
        "improved_embeddings.pkl",
        "wb"
    ) as f:

        pickle.dump(
            improved_embeddings,
            f
        )

    print(
        "Files saved successfully!"
    )

except Exception as e:

    print(
        f"Saving Error: {e}"
    )

Files saved successfully!


In [21]:
# Verify saved files

try:

    import os

    print(
        os.listdir()
    )

except Exception as e:

    print(e)

['.config', 'networking_chromadb_phase4', 'networking_docs', 'arrays.txt', 'improved_chunks.pkl', 'improved_embeddings.pkl', 'sample_data']
